# Model Gate — Uzbek Car Recognizer

Trains and compares models on the leakage-safe split from the Data Gate.

**Order the rubric rewards:** baselines → 2 real approaches → pick the best **on validation** →
evaluate **once** on the sealed test set → save reloadable artifacts.

**Before running:** `dataset_split.zip` (from the Data Gate) must be in Drive `CapstoneCars/`,
and set **Runtime → T4 GPU**. Full run ≈ 30–45 min; you can run the model cells one at a time.

> **The test set is not touched until Cell 10.** Choosing a model by peeking at test is the
> classic way to fool yourself — we select on `val/` only.

In [ ]:
# ── Cell 1 · Setup ──────────────────────────────────────────────
!pip install -q timm mlflow
import torch, torch.nn as nn, numpy as np, random, json
from pathlib import Path
from torch.amp import autocast, GradScaler
import timm
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = torch.cuda.is_available()
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("device:", dev, "| timm", timm.__version__)

In [ ]:
# ── Cell 2 · Load the leakage-safe split from Drive ─────────────
from google.colab import drive
drive.mount('/content/drive')
import glob
!rm -rf /content/dataset
zips = glob.glob('/content/drive/MyDrive/**/dataset_split.zip', recursive=True)
print("found:", zips)
ZIP = zips[0]
!unzip -q "$ZIP" -d /content
DATA = Path('/content/dataset')
for split in ['train','val','test']:
    print(split, {d.name: len(list(d.glob('*'))) for d in sorted((DATA/split).iterdir())})

In [ ]:
# ── Cell 3 · Dataloaders + class weights + label map ────────────
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMG = 224
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)   # ImageNet stats (all our backbones use them)
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

BS = 32
train_ds = datasets.ImageFolder(DATA/'train', train_tf)
val_ds   = datasets.ImageFolder(DATA/'val',   eval_tf)
test_ds  = datasets.ImageFolder(DATA/'test',  eval_tf)
train_loader = DataLoader(train_ds, BS, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   BS, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  BS, shuffle=False, num_workers=2, pin_memory=True)

CLASSES = train_ds.classes                 # alphabetical: cobalt, damas, gentra, nexia3, spark
class_to_idx = train_ds.class_to_idx
counts = np.bincount([y for _,y in train_ds.samples], minlength=len(CLASSES))
weights = torch.tensor(counts.sum()/(len(CLASSES)*counts), dtype=torch.float32).to(dev)
print("classes:", CLASSES)
print("train counts:", dict(zip(CLASSES, counts.tolist())))
print("class weights:", weights.cpu().numpy().round(3))

In [ ]:
# ── Cell 4 · MLflow + shared train/eval helpers ─────────────────
import os, mlflow
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'   # 2026 MLflow: opt into the file-based tracking store
mlflow.set_tracking_uri('file:/content/mlruns')
mlflow.set_experiment('uzbek-car-modelgate')
results, models = {}, {}   # filled by each model below

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for x,y in loader:
        with autocast('cuda', enabled=USE_AMP):
            out = model(x.to(dev))
        ps.append(out.argmax(1).cpu()); ys.append(y)
    y = torch.cat(ys).numpy(); p = torch.cat(ps).numpy()
    return {'acc': float(accuracy_score(y,p)),
            'macro_f1': float(f1_score(y,p,average='macro'))}, y, p

def train_model(model, epochs, lr_groups, tag, patience=3):
    model.to(dev)
    opt = torch.optim.AdamW(lr_groups, weight_decay=0.05)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    lossf = nn.CrossEntropyLoss(weight=weights)
    scaler = GradScaler('cuda', enabled=USE_AMP)
    best_f1, best_state, bad = -1.0, None, 0
    for ep in range(epochs):
        model.train(); tot = 0.0
        for x,y in train_loader:
            x,y = x.to(dev), y.to(dev)
            opt.zero_grad()
            with autocast('cuda', enabled=USE_AMP):
                loss = lossf(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()*x.size(0)
        sched.step()
        m,_,_ = evaluate(model, val_loader)
        print(f"[{tag}] ep {ep+1}/{epochs}  loss {tot/len(train_ds):.3f}  "
              f"val_acc {m['acc']:.3f}  val_macroF1 {m['macro_f1']:.3f}")
        if m['macro_f1'] > best_f1:
            best_f1 = m['macro_f1']
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state)
    return model, best_f1

def log_run(name, params, metrics):
    with mlflow.start_run(run_name=name):
        mlflow.log_params(params)
        mlflow.log_metrics({'val_'+k: v for k,v in metrics.items()})

In [ ]:
# ── Cell 5 · Baseline 1 · Majority class (the trivial floor) ────
from collections import Counter
maj = Counter([y for _,y in train_ds.samples]).most_common(1)[0][0]
yval = np.array([y for _,y in val_ds.samples])
results['majority'] = {'acc': float((yval==maj).mean()),
                       'macro_f1': float(f1_score(yval, np.full_like(yval, maj), average='macro'))}
log_run('baseline-majority', {'strategy': f'always {CLASSES[maj]}'}, results['majority'])
print(f"majority ({CLASSES[maj]}):", results['majority'])

In [ ]:
# ── Cell 6 · Baseline 2 · From-scratch CNN (why pretraining matters) ──
class SmallCNN(nn.Module):
    def __init__(self, n):
        super().__init__()
        def blk(i,o): return nn.Sequential(nn.Conv2d(i,o,3,padding=1), nn.BatchNorm2d(o),
                                           nn.ReLU(), nn.MaxPool2d(2))
        self.body = nn.Sequential(blk(3,32), blk(32,64), blk(64,128), blk(128,256),
                                  nn.AdaptiveAvgPool2d(1))
        self.head = nn.Linear(256, n)
    def forward(self, x): return self.head(self.body(x).flatten(1))

cnn = SmallCNN(len(CLASSES))
cnn, _ = train_model(cnn, epochs=6, lr_groups=[{'params':cnn.parameters(),'lr':1e-3}], tag='scratch-cnn')
results['scratch_cnn'], models['scratch_cnn'] = evaluate(cnn, val_loader)[0], cnn
log_run('baseline-scratch-cnn', {'arch':'small_cnn','epochs':6}, results['scratch_cnn'])
print("scratch CNN val:", results['scratch_cnn'])

In [ ]:
# ── Cell 7 · Approach A (primary) · ConvNeXt-Tiny fine-tune ──────
# Two phases: (1) warm up the new head on the frozen backbone, (2) unfreeze with
# discriminative LRs (low for backbone, higher for head).
cvx = timm.create_model('convnext_tiny.fb_in22k_ft_in1k', pretrained=True, num_classes=len(CLASSES))
head_params = list(cvx.get_classifier().parameters())

# Phase 1 — head only
for p in cvx.parameters(): p.requires_grad = False
for p in head_params:      p.requires_grad = True
cvx, _ = train_model(cvx, epochs=3, lr_groups=[{'params':head_params,'lr':1e-3}], tag='convnext-p1')

# Phase 2 — everything, discriminative LRs
for p in cvx.parameters(): p.requires_grad = True
head_ids = {id(p) for p in head_params}
backbone = [p for p in cvx.parameters() if id(p) not in head_ids]
cvx, _ = train_model(cvx, epochs=8, tag='convnext-p2',
                     lr_groups=[{'params':backbone,'lr':3e-5},{'params':head_params,'lr':3e-4}])

results['convnext_ft'], models['convnext_ft'] = evaluate(cvx, val_loader)[0], cvx
log_run('convnext-tiny-finetune',
        {'arch':'convnext_tiny.fb_in22k_ft_in1k','phase1_ep':3,'phase2_ep':8,
         'lr_backbone':3e-5,'lr_head':3e-4}, results['convnext_ft'])
print("ConvNeXt-Tiny val:", results['convnext_ft'])

In [ ]:
# ── Cell 8 · Approach B · ResNet-50 fine-tune (different architecture) ──
rn = timm.create_model('resnet50.a1_in1k', pretrained=True, num_classes=len(CLASSES))
rn_head = list(rn.get_classifier().parameters())
rn_head_ids = {id(p) for p in rn_head}
rn_backbone = [p for p in rn.parameters() if id(p) not in rn_head_ids]
rn, _ = train_model(rn, epochs=6, tag='resnet50',
                    lr_groups=[{'params':rn_backbone,'lr':5e-5},{'params':rn_head,'lr':5e-4}])
results['resnet50_ft'], models['resnet50_ft'] = evaluate(rn, val_loader)[0], rn
log_run('resnet50-finetune', {'arch':'resnet50.a1_in1k','epochs':6}, results['resnet50_ft'])
print("ResNet-50 val:", results['resnet50_ft'])

In [ ]:
# ── Cell 9 · Compare on VALIDATION and pick the winner ──────────
import pandas as pd
tbl = pd.DataFrame(results).T[['acc','macro_f1']].sort_values('macro_f1', ascending=False)
print("VALIDATION comparison (sorted by macro-F1):")
print(tbl.round(4))
winner = next(n for n in tbl.index if n in models)     # best TRAINED model (majority has no model)
print("\nSelected model:", winner)

In [ ]:
# ── Cell 10 · SEALED test evaluation (winner only, once) + error analysis ──
best_model = models[winner]
tm, ytrue, ypred = evaluate(best_model, test_loader)
print(f"=== SEALED TEST — {winner} ===")
print(f"test_acc {tm['acc']:.3f}   test_macroF1 {tm['macro_f1']:.3f}")
print(classification_report(ytrue, ypred, target_names=CLASSES, digits=3))

import matplotlib.pyplot as plt
cm = confusion_matrix(ytrue, ypred)
fig, ax = plt.subplots(figsize=(5,4)); ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha='right')
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i,j], ha='center', va='center', fontsize=8)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(f'{winner} — test confusion')
plt.tight_layout(); plt.savefig('/content/confusion.png', dpi=120); plt.show()

with mlflow.start_run(run_name=f'{winner}-TEST'):
    mlflow.log_metrics({'test_acc':tm['acc'],'test_macro_f1':tm['macro_f1']})
    mlflow.log_artifact('/content/confusion.png')

In [ ]:
# ── Cell 11 · Save reloadable artifacts + clean reload check ────
ART = Path('/content/artifacts'); ART.mkdir(exist_ok=True)
torch.save(best_model.state_dict(), ART/'model.pt')
MODEL_ID = {'convnext_ft':'convnext_tiny.fb_in22k_ft_in1k',
            'resnet50_ft':'resnet50.a1_in1k', 'scratch_cnn':'small_cnn'}[winner]
cfg = {'winner':winner, 'model_id':MODEL_ID, 'img_size':IMG,
       'mean':MEAN, 'std':STD, 'class_to_idx':class_to_idx, 'classes':CLASSES}
json.dump(cfg, open(ART/'config.json','w'), indent=2)

def build_from_cfg(cfg):
    m = SmallCNN(len(cfg['classes'])) if cfg['winner']=='scratch_cnn' else \
        timm.create_model(cfg['model_id'], pretrained=False, num_classes=len(cfg['classes']))
    m.load_state_dict(torch.load(ART/'model.pt', map_location=dev))
    return m.to(dev).eval()

from PIL import Image
reloaded = build_from_cfg(json.load(open(ART/'config.json')))
path, true = test_ds.samples[0]
x = eval_tf(Image.open(path).convert('RGB')).unsqueeze(0).to(dev)
with torch.no_grad(), autocast('cuda', enabled=USE_AMP):
    pred = reloaded(x).argmax(1).item()
print("reload OK — predicted:", CLASSES[pred], "| true:", CLASSES[true])
print("artifacts:", [p.name for p in ART.iterdir()])

In [ ]:
# ── Cell 12 · Back up artifacts + MLflow runs to Drive ──────────
!cd /content && zip -q -r modelgate_artifacts.zip artifacts mlruns confusion.png >/dev/null
!cp /content/modelgate_artifacts.zip /content/drive/MyDrive/CapstoneCars/
print("✅ model + config + MLflow runs backed up to Drive")